# 03 - SECOP 2025 | Texto no estructurado y detección de temas

Este notebook desarrolla la **Actividad 3: Texto no estructurado**.

## Objetivo

Crear una variable de texto consolidada llamada `texto_busqueda` y una variable analítica llamada `temas_detectados`, a partir de reglas de palabras clave aplicadas sobre la información textual de los contratos.

## Resultado esperado

Al finalizar este notebook se generan:

1. **Tabla con contratos y temas detectados.**
2. **Resumen de contratos por tema.**
3. **Explicación de reglas y limitaciones.**
4. **Manifiesto de trazabilidad de la Actividad 3.**

## Entrada principal

Este notebook usa como entrada preferente la tabla integrada creada en la actividad anterior:

```text
workspace.default.gold_secop_contratos_integrados
```

Si esa tabla no existe, intenta usar:

```text
workspace.default.silver_secop_ii_contratos
```

## Salidas principales

```text
workspace.default.gold_secop_contratos_temas
workspace.default.gold_resumen_contratos_por_tema
workspace.default.qa_reglas_temas_detectados
workspace.default.qa_activity_3_summary
```

## 1. Configuración general

En esta sección se definen el catálogo, esquema, rutas y nombres de tablas.

La lógica del notebook está diseñada para trabajar en Databricks con Spark y Delta Lake.

In [0]:
from datetime import datetime, timezone
import json

from pyspark.sql import functions as F
from pyspark.sql import types as T

# ============================================================
# CONFIGURACIÓN GENERAL
# ============================================================

BASE_PATH = "/Volumes/workspace/default/tallerspark/secop"

CATALOG = "workspace"
SCHEMA = "default"

# Entrada preferida: Gold integrado de la Actividad 2
TABLE_GOLD_INTEGRADA = f"{CATALOG}.{SCHEMA}.gold_secop_contratos_integrados"

# Entrada alternativa: Silver contratos
TABLE_SILVER_CONTRATOS = f"{CATALOG}.{SCHEMA}.silver_secop_ii_contratos"

# Salidas Actividad 3
TABLE_CONTRATOS_TEMAS = f"{CATALOG}.{SCHEMA}.gold_secop_contratos_temas"
TABLE_RESUMEN_TEMA = f"{CATALOG}.{SCHEMA}.gold_resumen_contratos_por_tema"
TABLE_REGLAS_TEMAS = f"{CATALOG}.{SCHEMA}.qa_reglas_temas_detectados"
TABLE_ACTIVITY_3_SUMMARY = f"{CATALOG}.{SCHEMA}.qa_activity_3_summary"

OUTPUT_PATH_GOLD = f"{BASE_PATH}/gold/secop_contratos_temas"
OUTPUT_PATH_MANIFEST = f"{BASE_PATH}/manifest"

print("BASE_PATH:", BASE_PATH)
print("Tabla entrada preferida:", TABLE_GOLD_INTEGRADA)
print("Tabla salida contratos-temas:", TABLE_CONTRATOS_TEMAS)
print("Tabla salida resumen por tema:", TABLE_RESUMEN_TEMA)

## 2. Lectura de la tabla base

El proceso intenta leer primero la tabla Gold integrada. Esta tabla es la mejor entrada porque ya contiene contratos, adiciones, ejecución y cruce territorial.

Si la tabla Gold no existe, el notebook usa como respaldo la tabla Silver de contratos.

Esto permite ejecutar la Actividad 3 aunque la integración Gold aún no esté completamente disponible.

In [0]:
def table_exists(table_name):
    try:
        spark.table(table_name).limit(1).count()
        return True
    except Exception:
        return False


if table_exists(TABLE_GOLD_INTEGRADA):
    df_base = spark.table(TABLE_GOLD_INTEGRADA)
    INPUT_TABLE_USED = TABLE_GOLD_INTEGRADA
    INPUT_LAYER = "gold"
else:
    df_base = spark.table(TABLE_SILVER_CONTRATOS)
    INPUT_TABLE_USED = TABLE_SILVER_CONTRATOS
    INPUT_LAYER = "silver_fallback"

print("Tabla usada como entrada:", INPUT_TABLE_USED)
print("Capa usada:", INPUT_LAYER)
print("Filas:", df_base.count())
print("Columnas:", len(df_base.columns))

display(df_base.limit(5))

## 3. Identificación de columnas textuales disponibles

Para crear `texto_busqueda`, el notebook identifica columnas textuales relevantes.

Se priorizan campos como:

- Objeto contractual.
- Descripción del proceso.
- Entidad.
- Proveedor.
- Sector.
- Tipo de contrato.
- Modalidad.
- Estado.
- Descripciones de adiciones o ejecución, si están disponibles.

No todos los datasets tienen exactamente las mismas columnas, por eso el código verifica cuáles existen antes de usarlas.

In [0]:
# ============================================================
# COLUMNAS CANDIDATAS PARA CONSTRUIR texto_busqueda
# ============================================================

TEXT_CANDIDATES = [
    "objeto_contrato_std",
    "descripcion_del_proceso",
    "detalle_del_objeto_a_contratar",
    "objeto_del_contrato",
    "nombre_entidad_std",
    "proveedor_std",
    "sector_std",
    "tipo_contrato_std",
    "modalidad_contratacion_std",
    "estado_contrato_std",
    "departamento_std",
    "ciudad_std",
    "ultima_descripcion_ejecucion_std",
    "descripcion_ejecucion_std",
    "descripcion_adicion_std"
]

available_text_columns = [c for c in TEXT_CANDIDATES if c in df_base.columns]

# Si no se encuentra ninguna candidata, toma columnas string con sufijo _std
if not available_text_columns:
    available_text_columns = [
        field.name for field in df_base.schema.fields
        if isinstance(field.dataType, T.StringType) and field.name.endswith("_std")
    ]

print("Columnas usadas para texto_busqueda:")
for c in available_text_columns:
    print("-", c)

if not available_text_columns:
    raise ValueError("No se encontraron columnas textuales para construir texto_busqueda.")

## 4. Creación de `texto_busqueda`

`texto_busqueda` es una columna que concatena varios campos textuales del contrato en una sola cadena.

La idea es tener un texto integrado donde se puedan buscar palabras clave asociadas a temas.

Ejemplo conceptual:

```text
objeto del contrato + entidad + sector + modalidad + descripción de ejecución
```

Luego se normaliza para facilitar la detección:

- Conversión a minúscula.
- Eliminación básica de tildes.
- Limpieza de espacios repetidos.
- Reemplazo de valores nulos por texto vacío.

In [0]:
# ============================================================
# CREACIÓN DE texto_busqueda
# ============================================================

def normalize_text_column(col_expr):
    text = F.lower(F.coalesce(col_expr.cast("string"), F.lit("")))

    # Normalización básica de tildes y caracteres frecuentes en español
    text = F.translate(
        text,
        "áéíóúüñÁÉÍÓÚÜÑ",
        "aeiouunAEIOUUN"
    )

    text = F.regexp_replace(text, r"[^a-zA-Z0-9\s]", " ")
    text = F.regexp_replace(text, r"\s+", " ")
    text = F.trim(text)

    return text


concat_expr = F.concat_ws(
    " ",
    *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in available_text_columns]
)

df_texto = (
    df_base
    .withColumn("texto_busqueda_original", concat_expr)
    .withColumn("texto_busqueda", normalize_text_column(F.col("texto_busqueda_original")))
    .withColumn("longitud_texto_busqueda", F.length(F.col("texto_busqueda")))
)

print("Filas con texto_busqueda:", df_texto.count())

display(df_texto.select(
    *([c for c in ["id_contrato_std", "nombre_entidad_std", "objeto_contrato_std"] if c in df_texto.columns]),
    "texto_busqueda",
    "longitud_texto_busqueda"
).limit(10))

## 5. Definición de reglas de temas

La detección de temas se realiza mediante reglas basadas en palabras clave.

Cada tema tiene:

- Nombre del tema.
- Expresión regular.
- Descripción.
- Limitación.

Este método es interpretable y fácil de explicar, pero tiene limitaciones: depende de las palabras definidas, puede generar falsos positivos y no entiende contexto profundo como lo haría un modelo NLP avanzado.

In [0]:
# ============================================================
# REGLAS DE DETECCIÓN DE TEMAS
# ============================================================

TOPIC_RULES = [
    {
        "tema": "infraestructura_obras",
        "regex": r"\b(obra|obras|construccion|construir|mejoramiento|adecuacion|mantenimiento vial|via|vias|puente|pavimento|placa huella|urbanismo|alcantarillado|acueducto|edificacion|infraestructura)\b",
        "descripcion": "Contratos relacionados con obras civiles, vías, construcción, mantenimiento o infraestructura física.",
        "limitacion": "Puede clasificar mantenimientos no relacionados con infraestructura si la descripción es ambigua."
    },
    {
        "tema": "salud",
        "regex": r"\b(salud|hospital|clinica|medico|medica|medicamento|medicamentos|ambulancia|eps|ips|paciente|vacuna|vacunacion|laboratorio clinico|biomedico|biomedica)\b",
        "descripcion": "Contratos asociados a servicios de salud, medicamentos, atención médica o equipamiento sanitario.",
        "limitacion": "Puede omitir contratos de salud si usan términos técnicos no incluidos en la regla."
    },
    {
        "tema": "educacion",
        "regex": r"\b(educacion|educativo|educativa|colegio|escuela|institucion educativa|universidad|estudiante|estudiantes|docente|docentes|matricula|aula|biblioteca escolar|formacion academica)\b",
        "descripcion": "Contratos relacionados con educación, instituciones educativas, formación o población estudiantil.",
        "limitacion": "Puede confundir capacitación empresarial con educación formal."
    },
    {
        "tema": "tecnologia_software",
        "regex": r"\b(software|hardware|sistema de informacion|sistemas de informacion|plataforma|licencia|licencias|aplicativo|aplicacion|tecnologia|tecnologico|servidor|nube|cloud|base de datos|soporte tecnico|mesa de ayuda|hosting|pagina web|desarrollo web)\b",
        "descripcion": "Contratos de software, licenciamiento, plataformas, infraestructura tecnológica o soporte TIC.",
        "limitacion": "Puede no detectar tecnologías descritas con marcas específicas si no aparece una palabra clave general."
    },
    {
        "tema": "alimentacion",
        "regex": r"\b(alimentacion|alimentos|comida|comedor|restaurante|racion|raciones|mercado|mercados|nutricion|refrigerio|refrigerios|pae|programa de alimentacion escolar)\b",
        "descripcion": "Contratos relacionados con suministro de alimentos, comedores, refrigerios o programas de alimentación.",
        "limitacion": "Puede clasificar contratos de eventos si mencionan refrigerios de forma secundaria."
    },
    {
        "tema": "transporte",
        "regex": r"\b(transporte|vehiculo|vehiculos|camioneta|bus|buses|combustible|gasolina|acpm|diesel|conductor|conductores|flota|moto|motocicleta|pasajes|tiquetes)\b",
        "descripcion": "Contratos de transporte, vehículos, combustible, conductores o movilidad.",
        "limitacion": "Puede mezclar transporte de personas, carga y suministro de combustible en un mismo tema amplio."
    },
    {
        "tema": "ambiente",
        "regex": r"\b(ambiente|ambiental|medio ambiente|residuos|reciclaje|forestal|reforestacion|conservacion|sostenible|sostenibilidad|agua|cuenca|fauna|flora|biodiversidad|ecosistema|vertimiento|emisiones)\b",
        "descripcion": "Contratos relacionados con gestión ambiental, recursos naturales, residuos, agua o sostenibilidad.",
        "limitacion": "Puede clasificar contratos que mencionan ambiente solo como requisito normativo."
    },
    {
        "tema": "seguridad_vigilancia",
        "regex": r"\b(seguridad|vigilancia|vigilante|vigilantes|cctv|camara|camaras|control de acceso|alarma|alarmas|monitoreo|proteccion|escolta)\b",
        "descripcion": "Contratos de vigilancia, seguridad física, cámaras, alarmas o control de acceso.",
        "limitacion": "No distingue entre seguridad física, seguridad informática y seguridad ciudadana si el texto es general."
    },
    {
        "tema": "servicios_profesionales",
        "regex": r"\b(prestacion de servicios|servicios profesionales|apoyo a la gestion|contratista|asesoria|asesor|consultoria|consultor|profesional especializado|honorarios)\b",
        "descripcion": "Contratos de prestación de servicios, consultoría, apoyo a la gestión o asesoría.",
        "limitacion": "Es un tema amplio y puede agrupar contratos de naturaleza muy diferente."
    },
    {
        "tema": "mantenimiento",
        "regex": r"\b(mantenimiento|mantener|reparacion|reparar|preventivo|correctivo|adecuacion|soporte|calibracion|instalacion)\b",
        "descripcion": "Contratos de mantenimiento preventivo, correctivo, reparación, calibración o soporte.",
        "limitacion": "Puede cruzarse con infraestructura, tecnología o equipos biomédicos."
    },
    {
        "tema": "aseo_limpieza",
        "regex": r"\b(aseo|limpieza|desinfeccion|fumigacion|lavado|servicios generales|jardineria|cafeteria|higiene)\b",
        "descripcion": "Contratos de aseo, limpieza, desinfección, fumigación, jardinería o servicios generales.",
        "limitacion": "Puede no detectar contratos donde se use una marca o descripción indirecta del servicio."
    },
    {
        "tema": "cultura_deporte",
        "regex": r"\b(cultura|cultural|deporte|deportivo|recreacion|recreativo|evento|eventos|festival|artista|musica|biblioteca|ludica|turismo)\b",
        "descripcion": "Contratos relacionados con cultura, deporte, recreación, eventos o turismo.",
        "limitacion": "Eventos institucionales pueden clasificarse aquí aunque su objetivo principal sea administrativo."
    },
    {
        "tema": "suministros_compras",
        "regex": r"\b(suministro|suministros|compra|adquisicion|dotacion|elementos|insumos|materiales|papeleria|mobiliario|equipos|herramientas)\b",
        "descripcion": "Contratos de adquisición, suministro, dotación, materiales, insumos o equipos.",
        "limitacion": "Es una categoría transversal y puede solaparse con salud, educación, tecnología o mantenimiento."
    },
    {
        "tema": "interventoria_supervision",
        "regex": r"\b(interventoria|interventor|supervision|supervisor|seguimiento tecnico|control tecnico|auditoria|auditor)\b",
        "descripcion": "Contratos de interventoría, supervisión, auditoría o seguimiento técnico.",
        "limitacion": "Puede clasificar auditorías administrativas, técnicas o financieras en un mismo grupo."
    },
    {
        "tema": "juridico_administrativo",
        "regex": r"\b(juridico|juridica|legal|abogado|abogada|defensa judicial|representacion judicial|administrativo|administrativa|gestion documental|archivo|correspondencia)\b",
        "descripcion": "Contratos relacionados con apoyo jurídico, administrativo, documental o legal.",
        "limitacion": "Puede incluir apoyos administrativos muy diferentes entre sí."
    }
]

df_topic_rules = spark.createDataFrame(TOPIC_RULES)

display(df_topic_rules)

(
    df_topic_rules.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE_REGLAS_TEMAS)
)

print("Tabla de reglas guardada:", TABLE_REGLAS_TEMAS)

## 6. Aplicación de reglas y creación de `temas_detectados`

En esta sección se aplican las reglas sobre `texto_busqueda`.

El resultado es una columna tipo arreglo:

```text
temas_detectados = ["salud", "tecnologia_software", ...]
```

Un contrato puede tener más de un tema, porque un mismo objeto contractual puede mencionar varios conceptos.  
Ejemplo: un contrato de software para un hospital puede quedar clasificado como:

```text
["salud", "tecnologia_software"]
```

Si no se detecta ningún tema, se asigna:

```text
["sin_tema_detectado"]
```

In [0]:
# ============================================================
# DETECCIÓN DE TEMAS
# ============================================================

topic_columns = []

df_topics = df_texto

for rule in TOPIC_RULES:
    tema = rule["tema"]
    regex = rule["regex"]
    flag_col = f"flag_tema_{tema}"

    df_topics = df_topics.withColumn(
        flag_col,
        F.when(F.col("texto_busqueda").rlike(regex), F.lit(True)).otherwise(F.lit(False))
    )

    topic_columns.append((tema, flag_col))

# Construir array de temas detectados
array_expr = F.array(*[
    F.when(F.col(flag_col) == True, F.lit(tema)).otherwise(F.lit(None))
    for tema, flag_col in topic_columns
])

df_topics = (
    df_topics
    .withColumn("temas_detectados_raw", array_expr)
    .withColumn("temas_detectados", F.expr("filter(temas_detectados_raw, x -> x is not null)"))
    .withColumn(
        "temas_detectados",
        F.when(
            F.size(F.col("temas_detectados")) == 0,
            F.array(F.lit("sin_tema_detectado"))
        ).otherwise(F.col("temas_detectados"))
    )
    .withColumn("numero_temas_detectados", F.size(F.col("temas_detectados")))
    .drop("temas_detectados_raw")
)

print("Filas clasificadas:", df_topics.count())

display(df_topics.select(
    *([c for c in ["id_contrato_std", "nombre_entidad_std", "objeto_contrato_std"] if c in df_topics.columns]),
    "texto_busqueda",
    "temas_detectados",
    "numero_temas_detectados"
).limit(20))

## 7. Tabla con contratos y temas

Esta tabla cumple el primer resultado esperado de la actividad:

```text
tabla con contratos y temas
```

Se seleccionan columnas clave para análisis:

- Identificador del contrato.
- Entidad.
- Proveedor.
- Objeto contractual.
- Departamento y ciudad.
- Valor del contrato.
- Número de adiciones.
- Último avance de ejecución.
- Texto de búsqueda.
- Temas detectados.

In [0]:
# ============================================================
# TABLA FINAL DE CONTRATOS CON TEMAS
# ============================================================

preferred_columns = [
    "id_contrato_std",
    "fecha_firma",
    "valor_contrato_num",
    "nombre_entidad_std",
    "proveedor_std",
    "sector_std",
    "tipo_contrato_std",
    "modalidad_contratacion_std",
    "estado_contrato_std",
    "departamento_std",
    "ciudad_std",
    "cod_depto_std",
    "cod_mpio_std",
    "tiene_cruce_territorial",
    "numero_adiciones",
    "valor_total_adiciones",
    "ultima_fecha_adicion",
    "ultima_fecha_ejecucion",
    "ultimo_avance_real_num",
    "prioridad_revision",
    "objeto_contrato_std",
    "texto_busqueda",
    "longitud_texto_busqueda",
    "temas_detectados",
    "numero_temas_detectados"
]

selected_columns = [c for c in preferred_columns if c in df_topics.columns]

# Se conservan también flags de temas para auditoría
flag_columns = [flag_col for _, flag_col in topic_columns if flag_col in df_topics.columns]

df_contratos_temas = (
    df_topics
    .select(*(selected_columns + flag_columns))
    .withColumn("_activity_3_created_at", F.current_timestamp())
)

print("Contratos con temas - filas:", df_contratos_temas.count())
print("Contratos con temas - columnas:", len(df_contratos_temas.columns))

display(df_contratos_temas.limit(20))

(
    df_contratos_temas.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE_CONTRATOS_TEMAS)
)

print("Tabla guardada:", TABLE_CONTRATOS_TEMAS)

## 8. Resumen de contratos por tema

Para generar el resumen, se explota la columna `temas_detectados`.

Esto significa que si un contrato tiene tres temas, aparecerá una vez por cada tema en la tabla de resumen.

El resultado incluye:

- Cantidad de contratos por tema.
- Valor total de contratos por tema.
- Valor promedio de contratos.
- Número promedio de adiciones.
- Distribución porcentual de contratos.

In [0]:
# ============================================================
# RESUMEN DE CONTRATOS POR TEMA
# ============================================================

df_exploded_topics = (
    df_contratos_temas
    .withColumn("tema_detectado", F.explode(F.col("temas_detectados")))
)

total_contracts = df_contratos_temas.count()

summary_aggs = [
    F.countDistinct("id_contrato_std").alias("contratos_distintos"),
    F.count("*").alias("registros_tema"),
]

if "valor_contrato_num" in df_contratos_temas.columns:
    summary_aggs.extend([
        F.sum("valor_contrato_num").alias("valor_total_contratos"),
        F.avg("valor_contrato_num").alias("valor_promedio_contrato"),
        F.max("valor_contrato_num").alias("valor_maximo_contrato")
    ])

if "numero_adiciones" in df_contratos_temas.columns:
    summary_aggs.append(F.avg("numero_adiciones").alias("promedio_adiciones"))

df_resumen_tema = (
    df_exploded_topics
    .groupBy("tema_detectado")
    .agg(*summary_aggs)
    .withColumn(
        "porcentaje_contratos",
        F.round((F.col("contratos_distintos") / F.lit(total_contracts)) * 100, 4)
    )
    .orderBy(F.desc("contratos_distintos"))
)

display(df_resumen_tema)

(
    df_resumen_tema.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE_RESUMEN_TEMA)
)

print("Tabla guardada:", TABLE_RESUMEN_TEMA)

## 9. Revisión de contratos sin tema detectado

Esta sección permite revisar los contratos que no fueron clasificados por ninguna regla.

Esto es importante porque muestra una limitación del enfoque basado en palabras clave.  
Si aparecen demasiados contratos sin tema, se deben ampliar las reglas o mejorar el enfoque de clasificación.

In [0]:
df_sin_tema = (
    df_contratos_temas
    .filter(F.array_contains(F.col("temas_detectados"), "sin_tema_detectado"))
)

sin_tema_count = df_sin_tema.count()
total_count = df_contratos_temas.count()
sin_tema_pct = round((sin_tema_count / total_count) * 100, 4) if total_count else None

print("Total contratos:", total_count)
print("Contratos sin tema detectado:", sin_tema_count)
print("Porcentaje sin tema detectado:", sin_tema_pct)

display(df_sin_tema.select(
    *([c for c in ["id_contrato_std", "nombre_entidad_std", "objeto_contrato_std", "texto_busqueda"] if c in df_sin_tema.columns]),
    "temas_detectados"
).limit(50))

## 10. Revisión de contratos con múltiples temas

Un contrato puede tener varios temas detectados.  
Esto no es necesariamente un error; puede indicar que el objeto contractual es transversal.

Ejemplo:

```text
Contrato de licenciamiento de software para una entidad de salud
→ temas: tecnologia_software, salud
```

Esta sección permite revisar esos casos.

In [0]:
df_multi_tema = (
    df_contratos_temas
    .filter(F.col("numero_temas_detectados") > 1)
)

multi_count = df_multi_tema.count()
multi_pct = round((multi_count / total_count) * 100, 4) if total_count else None

print("Contratos con múltiples temas:", multi_count)
print("Porcentaje con múltiples temas:", multi_pct)

display(df_multi_tema.select(
    *([c for c in ["id_contrato_std", "nombre_entidad_std", "objeto_contrato_std", "texto_busqueda"] if c in df_multi_tema.columns]),
    "temas_detectados",
    "numero_temas_detectados"
).limit(50))

## 11. Resumen de calidad de texto

Se revisan condiciones básicas de calidad sobre `texto_busqueda`:

- Textos nulos.
- Textos vacíos.
- Longitud mínima.
- Longitud máxima.
- Longitud promedio.
- Contratos sin tema.
- Contratos con múltiples temas.

In [0]:
text_quality = {
    "total_contracts": total_count,
    "null_texto_busqueda": df_contratos_temas.filter(F.col("texto_busqueda").isNull()).count(),
    "empty_texto_busqueda": df_contratos_temas.filter(F.trim(F.col("texto_busqueda")) == "").count(),
    "sin_tema_detectado_count": sin_tema_count,
    "sin_tema_detectado_pct": sin_tema_pct,
    "multi_tema_count": multi_count,
    "multi_tema_pct": multi_pct,
    "created_at_utc": datetime.now(timezone.utc).isoformat()
}

length_stats = (
    df_contratos_temas
    .agg(
        F.min("longitud_texto_busqueda").alias("min_length"),
        F.max("longitud_texto_busqueda").alias("max_length"),
        F.avg("longitud_texto_busqueda").alias("avg_length")
    )
    .collect()[0]
)

text_quality.update({
    "min_text_length": int(length_stats["min_length"]) if length_stats["min_length"] is not None else None,
    "max_text_length": int(length_stats["max_length"]) if length_stats["max_length"] is not None else None,
    "avg_text_length": float(length_stats["avg_length"]) if length_stats["avg_length"] is not None else None,
})

df_text_quality = spark.createDataFrame([text_quality])
display(df_text_quality)

(
    df_text_quality.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.qa_texto_busqueda_calidad")
)

print("Tabla guardada:", f"{CATALOG}.{SCHEMA}.qa_texto_busqueda_calidad")

## 12. Guardar salidas en Parquet

Además de guardar tablas Delta, se guardan copias en Parquet dentro del Volume del proyecto.

In [0]:
topics_output_path = f"{OUTPUT_PATH_GOLD}/contratos_temas"
summary_output_path = f"{OUTPUT_PATH_GOLD}/resumen_por_tema"
rules_output_path = f"{OUTPUT_PATH_GOLD}/reglas_temas"

(
    df_contratos_temas.write
    .mode("overwrite")
    .parquet(topics_output_path)
)

(
    df_resumen_tema.write
    .mode("overwrite")
    .parquet(summary_output_path)
)

(
    df_topic_rules.write
    .mode("overwrite")
    .parquet(rules_output_path)
)

print("Parquet contratos-temas:", topics_output_path)
print("Parquet resumen-temas:", summary_output_path)
print("Parquet reglas:", rules_output_path)

## 13. Manifiesto de la Actividad 3

Se guarda un manifiesto JSON con:

- Tabla de entrada usada.
- Tablas de salida.
- Número de reglas.
- Número de contratos procesados.
- Número de contratos sin tema.
- Limitaciones principales.

In [0]:
activity_3_manifest = {
    "project": "SECOP Big Data 2025",
    "activity": "Actividad 3 - Texto no estructurado",
    "input_table_used": INPUT_TABLE_USED,
    "input_layer": INPUT_LAYER,
    "output_tables": {
        "contracts_topics": TABLE_CONTRATOS_TEMAS,
        "summary_by_topic": TABLE_RESUMEN_TEMA,
        "topic_rules": TABLE_REGLAS_TEMAS,
        "text_quality": f"{CATALOG}.{SCHEMA}.qa_texto_busqueda_calidad"
    },
    "output_paths": {
        "contracts_topics_parquet": topics_output_path,
        "summary_by_topic_parquet": summary_output_path,
        "rules_parquet": rules_output_path
    },
    "available_text_columns": available_text_columns,
    "number_of_topic_rules": len(TOPIC_RULES),
    "total_contracts": total_count,
    "sin_tema_detectado_count": sin_tema_count,
    "sin_tema_detectado_pct": sin_tema_pct,
    "multi_tema_count": multi_count,
    "multi_tema_pct": multi_pct,
    "main_limitations": [
        "La clasificación se basa en reglas de palabras clave, no en comprensión semántica profunda.",
        "Un contrato puede pertenecer a varios temas si contiene palabras de varias categorías.",
        "Pueden existir falsos positivos cuando una palabra aparece en un contexto secundario.",
        "Pueden existir falsos negativos si el contrato usa sinónimos o términos no incluidos en las reglas.",
        "La calidad de la clasificación depende de la calidad del texto original del SECOP."
    ],
    "created_at_utc": datetime.now(timezone.utc).isoformat()
}

manifest_path = f"{OUTPUT_PATH_MANIFEST}/activity_3_text_topics_manifest.json"

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(activity_3_manifest, f, ensure_ascii=False, indent=4)

print("Manifest saved:", manifest_path)

## 14. Resumen final de cumplimiento

Esta sección crea una tabla QA que resume el cumplimiento de la Actividad 3.

In [0]:
activity_3_summary = [
    {
        "requirement": "Crear texto_busqueda",
        "result": "Columna texto_busqueda creada mediante concatenación y normalización de campos textuales.",
        "output": TABLE_CONTRATOS_TEMAS,
        "status": "OK"
    },
    {
        "requirement": "Crear temas_detectados",
        "result": "Columna temas_detectados creada mediante reglas de palabras clave y expresiones regulares.",
        "output": TABLE_CONTRATOS_TEMAS,
        "status": "OK"
    },
    {
        "requirement": "Tabla con contratos y temas",
        "result": "Tabla gold_secop_contratos_temas creada en Delta.",
        "output": TABLE_CONTRATOS_TEMAS,
        "status": "OK"
    },
    {
        "requirement": "Resumen de contratos por tema",
        "result": "Tabla gold_resumen_contratos_por_tema creada con conteos, valores y porcentajes.",
        "output": TABLE_RESUMEN_TEMA,
        "status": "OK"
    },
    {
        "requirement": "Explicación de reglas y limitaciones",
        "result": "Tabla qa_reglas_temas_detectados y manifiesto JSON creados con descripciones y limitaciones.",
        "output": TABLE_REGLAS_TEMAS,
        "status": "OK"
    }
]

df_activity_3_summary = spark.createDataFrame(activity_3_summary)

display(df_activity_3_summary)

(
    df_activity_3_summary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE_ACTIVITY_3_SUMMARY)
)

print("Tabla guardada:", TABLE_ACTIVITY_3_SUMMARY)

## 15. Texto sugerido para el informe

En la Actividad 3 se trabajó el componente de texto no estructurado de los contratos SECOP. Para ello se creó la variable `texto_busqueda`, consolidando campos descriptivos del contrato como objeto contractual, entidad, proveedor, sector, modalidad y descripciones disponibles. Posteriormente se aplicó un conjunto de reglas basadas en palabras clave y expresiones regulares para construir la variable `temas_detectados`. Cada contrato puede tener uno o varios temas asociados, y aquellos que no cumplen ninguna regla se clasifican como `sin_tema_detectado`.

Como resultado, se generó una tabla con contratos y temas, un resumen agregado de contratos por tema y una tabla documentada con las reglas utilizadas. La metodología es transparente e interpretable, pero tiene limitaciones: depende de la calidad del texto original, puede generar falsos positivos cuando una palabra aparece en un contexto secundario y puede omitir temas si el contrato usa términos no contemplados en las reglas. Por esta razón, los resultados deben interpretarse como una clasificación exploratoria basada en reglas y no como un modelo avanzado de procesamiento de lenguaje natural.